# ENPG, CASEN y oportunidades de contacto con APS

**FONDECYT 1240138 · actualizado el 27 de octubre de 2026.** Insumos independientes para el microsimulador.
El ajuste corrige composición observada bajo supuestos de transporte; no identifica todo el sesgo de consumo.
La exploración comunal distingue población residente, contacto sanitario declarado y utilización de APS.

Este notebook **lee y verifica resultados de una ejecución real de los microdatos** por defecto.
Para reconstruirlos, ejecutar con `ECA_REBUILD=1`; cada sección llama al mismo código que generó los archivos.
Las salidas guardadas incluyen tablas HTML y cuatro figuras con IRdisplay, sin JavaScript ni recursos externos.
Fuera de Jupyter se usa htmltools::browsable(), como en expand_pif2/3.
Las salidas guardadas no son una nueva ejecución integral del microsimulador. No se modificaron sus notebooks.
Los identificadores, pesos y filas individuales se conservan únicamente en bundles cifrados.

Código: [enpg_casen_aps.R](enpg_casen_aps.R) y [enpg_casen_rural_prelim.R](enpg_casen_rural_prelim.R).
Explicación: [enpg_casen_aps_explicado.html](enpg_casen_aps_explicado.html).


In [1]:
.t0 <- Sys.time()
source(here::here("__andres_control", "enpg_casen_aps.R"))
rebuild <- identical(Sys.getenv("ECA_REBUILD"), "1")
cat("Rebuild from microdata:", rebuild, "\n")
eca_tests()
read_result <- function(name) utils::read.csv(file.path(eca_dir, paste0(name, ".csv")))

# Same Jupyter / browsable dispatch as expand_pif2 and expand_pif3.
eca_show_table <- function(data, caption, digits = 3L) {
  tag <- htmltools::div(
    style = "max-height:520px;max-width:100%;overflow:auto;",
    tabindex = "0", role = "region", `aria-label` = caption,
    htmltools::tags$style(".eca-table{border-collapse:collapse;font-size:14px;width:100%}.eca-table th,.eca-table td{padding:6px 10px;border-bottom:1px solid #ccc}.eca-table caption{text-align:left;font-weight:bold;padding:8px 0}"),
    htmltools::HTML(knitr::kable(as.data.frame(data), format = "html",
      digits = digits, row.names = FALSE, caption = caption, escape = TRUE,
      table.attr = 'class="eca-table"')))
  if (isTRUE(getOption("jupyter.in_kernel"))) {
    IRdisplay::display_html(as.character(tag))
    invisible(tag)
  } else htmltools::browsable(tag)
}
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


Elapsed: 0.00 min
Rebuild from microdata: FALSE 
PASS: education, missing completion, missing education, totals, duplicate targets, unsupported cells
Elapsed: 0.00 min
Elapsed: 0.00 min


## 1. Reconstruir la referencia nacional

CASEN 2013–2024, población de 15–65 años; `expr_c2017_nm` en 2013/2015 y `expr` desde 2017.
Se reutiliza la regla `e3` del script anterior, sin ejecutar sus rutas ni sobrescribir sus salidas.
El diseño completo usa `varunit` y `varstrat` antes de seleccionar el dominio. Se ajustan los estratos con
una sola UPM en el dominio (`survey.lonely.psu='adjust'` y `survey.adjust.domain.lonely=TRUE`).
La concordancia con la variable del Ministerio valida la recodificación, no el transporte de consumo.

Las proporciones educativas excluyen educación desconocida. La distribución sexo–edad incluye a todas las
personas elegibles. Los años intermedios se interpolan linealmente; 2012 usa 2013, sin extrapolar una tendencia.
Las metas antiguas del repositorio se comparan con la reconstrucción: **no se toman como actualizadas por su nombre**.


In [2]:
.t0 <- Sys.time()
if (rebuild) eca_casen_targets()
eca_show_table(read_result("casen_education_validation"), "Validación educativa por ronda CASEN")
eca_show_table(read_result("casen_year_interpolation"), "Fuentes CASEN para cada año ENPG")
eca_show_table(read_result("casen_existing_targets_check"), "Comparación con las metas anteriores")
targets <- read_result("casen_targets_enpg_years")
stopifnot(nrow(targets) == 56, all(abs(rowSums(targets[c("e1", "e2", "e3")]) - 1) < 1e-8))
eca_show_table(targets, "Metas de sexo, edad y educación: todos los años")
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


anio,n,pct_missing,agreement_pct,population_15_65
2013,148209,0.282,98.663,12331761
2015,179759,0.062,99.778,12583858
2017,145890,0.297,99.780,12925292
2020,125536,0.000,99.630,13613690
2022,135584,0.000,99.725,13768889
2024,146764,0.000,99.877,13843287


anio,a,b,wb
2012,2013,2013,0.000
2014,2013,2015,0.500
2016,2015,2017,0.500
2018,2017,2020,0.333
2020,2020,2020,0.000
2022,2022,2022,0.000
2024,2024,2024,0.000


max_abs_difference_existing
0.02


sexo,ag,anio,e1,e2,e3,p_sexo_edad
1,15-29,2012,0.374,0.518,0.108,0.175
2,15-29,2012,0.334,0.538,0.128,0.170
1,30-44,2012,0.302,0.425,0.273,0.155
2,30-44,2012,0.285,0.443,0.272,0.155
1,45-59,2012,0.484,0.343,0.173,0.131
2,45-59,2012,0.501,0.342,0.157,0.137
1,60-65,2012,0.610,0.263,0.126,0.036
2,60-65,2012,0.642,0.222,0.136,0.041
1,15-29,2014,0.339,0.547,0.114,0.173
2,15-29,2014,0.308,0.553,0.139,0.168


Elapsed: 0.00 min


## 2. Armonizar educación del entrevistado y ajustar la ENPG

2012: `p184/p185`; 2014: `dp9/dp10`; 2016: `dp_9/dp_10`; 2018: `T_DP_12/T_DP_13`.
En esas cuatro olas se selecciona la columna del jefe de hogar **solo cuando el entrevistado es el jefe**.
Desde 2020 se usa `DP_12/DP_13`. En 2014/2016, las preguntas 12/13 se refieren a ingresos.
La finalización desconocida queda faltante, en lugar de tratarla automáticamente como nivel incompleto.

**Ajuste educativo:** `peso_educ = peso × p_CASEN(educación|sexo,edad) / p_ENPG(educación|sexo,edad)`.
El denominador usa solo educación conocida. Quien no tiene educación conocida conserva `peso`.
Por eso se conservan exactamente los totales originales de sexo–edad y las proporciones educativas se cumplen
entre quienes tienen educación conocida. La fórmula del contexto, con desconocidos dentro del denominador,
no conservaría los totales si luego se mantienen sus pesos.

**Ajuste nacional adicional:** `peso_nacional = peso_educ × p_CASEN(sexo,edad) / p_ENPG(sexo,edad)`.
Este sí cambia la distribución de sexo–edad y conserva el total de pesos ENPG de cada año. No convierte ese
total en un conteo nacional CASEN. Los casos sin educación también reciben el factor sexo–edad, pero no el educativo.
El supuesto de ausencia selectiva de educación no queda validado por estas igualdades.


In [3]:
.t0 <- Sys.time()
if (rebuild) { eca_enpg(); eca_national_weights() }
eca_show_table(read_result("enpg_variable_map"), "Variables originales por ola ENPG")
eca_show_table(read_result("enpg_audit"), "Muestra y educación faltante por ola ENPG")
eca_show_table(read_result("enpg_consumption_comparison"), "Consumo antes y después de ajustar los pesos", digits = 5)
eca_show_table(read_result("enpg_weight_diagnostics"), "Diagnóstico de pesos: ajuste educativo")
eca_show_table(read_result("enpg_national_weight_diagnostics"), "Diagnóstico de pesos: composición nacional")
totals <- read_result("enpg_total_checks")
national <- read_result("enpg_national_target_checks")
stopifnot(all(abs(totals$after / totals$before - 1) < 1e-8), all(abs(national$p - national$p_sexo_edad) < 1e-8))
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


anio,file,id,weight,sex,age,level,complete,head,head_level,head_complete,ever,last
2012,Base de datos ENPG 2012 (PG).DTA.dta,idencuesta,PONDERADOR,sexo,edad,p184_1,p185_1,p182,p184_2,p185_2,p10,p13
2014,Base de datos ENPG 2014 (PG).DTA.dta,idencuesta,F2_MAY_AJUS_com,sexo,edad,dp9_1,dp10_1,dp7,dp9_2,dp10_2,oh1,oh4
2016,base ENPG 2016 publico general.dta,idencuesta,Fexp,sexo,edad,dp_9_a,dp_10_a,dp_7,dp_9_b,dp_10_b,oh_1,oh_4
2018,Base de datos ENPG 2018 (PG).DTA,SbjNum,Fexp,S01,S02,T_DP_12_1,T_DP_13_1,DP_10,T_DP_12_2,T_DP_13_2,OH_1,OH_4
2020,enpg2020.RDS,SbjNum,FACT_PERS_COMUNA,S01,S02,DP_12,DP_13,NA,NA,NA,OH_1,OH_4
2022,BD - ENPG 2022 (Stata 16).dta,FOLIO,FACTOR_EXPANSION,SEXO,EDAD,DP_12,DP_13,NA,NA,NA,OH_1,OH_4
2024,Base Publica ENPG 2024 (Stata 16).dta,RESPONDENT_SERIAL,FACTOR_EXPANSION,SEXO,EDAD,DP_12,DP_13,NA,NA,NA,OH_1,OH_4


anio,n_raw,n_eligible,n_excluded,n_head_selected,n_educ_missing,pct_educ_missing_weighted,n_completion_unknown,n_alcohol_inconsistent
2012,17154,16322,832,7620,109,0.934,81,0
2014,20113,19326,787,9040,508,3.039,265,0
2016,19147,18354,793,8972,230,0.950,61,0
2018,19427,18734,693,9467,217,0.777,49,0
2020,16662,16114,548,0,105,0.470,38,0
2022,17454,17023,431,0,107,0.531,37,0
2024,18668,18231,437,0,132,0.789,57,0


anio,outcome,p_original,p_educ,p_nacional,change_national_pp
2012,current_30d,0.43267,0.42490,0.42983,-0.28382
2014,current_30d,0.51036,0.50234,0.50757,-0.27872
2016,current_30d,0.48368,0.47018,0.47402,-0.96522
2018,current_30d,0.45721,0.45468,0.46095,0.37358
2020,current_30d,0.46682,0.46381,0.46957,0.27485
2022,current_30d,0.40589,0.39744,0.40328,-0.26059
2024,current_30d,0.35946,0.35562,0.36025,0.07926
2012,current_12m,0.58757,0.58054,0.58602,-0.15500
2014,current_12m,0.65879,0.65011,0.65753,-0.12631
2016,current_12m,0.67046,0.65785,0.66324,-0.72136


anio,n,min_factor,max_factor,neff_original,neff_educ
2012,16322,0.684,1.275,4654.294,4793.641
2014,19326,0.690,1.338,4381.772,4386.986
2016,18354,0.686,1.447,4768.018,4721.000
2018,18734,0.721,1.344,4492.248,4510.310
2020,16114,0.793,1.365,3472.741,3533.164
2022,17023,0.620,1.515,8065.934,7955.516
2024,18231,0.722,1.508,4976.519,4955.677


anio,n,min_factor,max_factor,neff_original,neff_nacional
2012,16322,0.548,1.581,4654.294,4822.551
2014,19326,0.476,1.557,4381.772,4349.398
2016,18354,0.610,1.628,4768.018,4671.502
2018,18734,0.586,1.481,4492.248,4354.014
2020,16114,0.658,1.336,3472.741,3440.322
2022,17023,0.579,1.629,8065.934,7800.842
2024,18231,0.598,1.749,4976.519,4885.614


Elapsed: 0.00 min


Las comparaciones son prevalencia del último mes y del último año entre respuestas válidas, con los mismos
registros antes y después. Son estimaciones puntuales ponderadas; no se fabrican UPM o estratos ENPG ni se presentan
intervalos de encuesta sin una armonización completa del diseño. `neff_kish` mide dispersión de pesos, no incorpora
correlación intraclúster. La incertidumbre de las metas CASEN no se propagó a los resultados ENPG.

Los pesos se pueden unir al motor por `anio + id` usando el identificador original de cada ola. Antes de integrarlos
hay que verificar una unión uno a uno y la población analítica del motor; no se aplicaron a PIF/AAF en este trabajo.


## 3. Edad simple para inicialización; tramos para pesos

Se entregan perfiles por edad 15–65, promediados con las edades vecinas disponibles. En los extremos se usan dos
edades distintas; el código antiguo duplicaba la edad del extremo y dividía por tres. El archivo conserva la
procedencia anual por separado. Estos perfiles son cortes transversales: **sus diferencias no identifican
transiciones educativas individuales**. No se usa este archivo para crear 306 celdas de pesos ENPG.


In [4]:
.t0 <- Sys.time()
age <- read_result("casen_single_age_targets_enpg_years")
stopifnot(nrow(age) == 714, all(abs(rowSums(age[c("e1", "e2", "e3")]) - 1) < 1e-8))
eca_show_table(age |> dplyr::count(anio, name = "rows"), "Cobertura del insumo por edad simple", digits = 0)
eca_show_table(age[age$sexo == 1 & age$edad == 30, ], "Ejemplo: hombres de 30 años, serie completa")
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


anio,rows
2012,102
2014,102
2016,102
2018,102
2020,102
2022,102
2024,102


sexo,edad,anio,e1,e2,e3
1,30,2012,0.204,0.489,0.308
1,30,2014,0.175,0.502,0.323
1,30,2016,0.154,0.500,0.347
1,30,2018,0.147,0.470,0.383
1,30,2020,0.120,0.443,0.437
1,30,2022,0.128,0.472,0.400
1,30,2024,0.120,0.469,0.411


Elapsed: 0.00 min


## 4. Población por sexo, edad y comuna

Se reutilizan las proyecciones INE que ya usa `eps_reid.R`, sumando áreas urbana y rural.
La fuente es la serie comunal existente, basada en Censo 2017; no se presenta como una nueva serie comunal del Censo 2024.
Incluye 346 comunas y 2012–2024. Por sus edades quinquenales, esta tabla usa **15–64**, con último tramo 60–64.
No se inventa el número de personas de 65 años dividiendo el grupo 65–69.

Se entregan dos denominadores: `p_sexo_edad_en_comuna` suma uno dentro de comuna; `p_comuna_dado_sexo_edad`
suma uno entre comunas para cada sexo–edad. Ninguno es un porcentaje de uso de APS.


In [5]:
.t0 <- Sys.time()
if (rebuild) eca_demography()
population <- read_result("ine_sexo_edad_comuna_2012_2024")
stopifnot(nrow(population) == 19376)
eca_show_table(population |> dplyr::count(anio, name = "rows"), "Cobertura del insumo comunal", digits = 0)
eca_show_table(population[population$comuna == 13101 & population$sexo == 1 & population$ag == "30-44", ], "Ejemplo agregado: Santiago, hombres de 30–44 años")
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


anio,rows
2012,2768
2014,2768
2016,2768
2018,2768
2020,2768
2022,2768
2024,2768


anio,region,comuna,nombre,sexo,ag,population,p_sexo_edad_en_comuna,p_comuna_dado_sexo_edad
2012,13,13101,Santiago,1,30-44,57469,0.201,0.030
2014,13,13101,Santiago,1,30-44,63285,0.206,0.033
2016,13,13101,Santiago,1,30-44,70826,0.212,0.036
2018,13,13101,Santiago,1,30-44,83052,0.221,0.040
2020,13,13101,Santiago,1,30-44,93357,0.230,0.042
2022,13,13101,Santiago,1,30-44,101440,0.240,0.044
2024,13,13101,Santiago,1,30-44,107276,0.247,0.045


Elapsed: 0.00 min


## 5. Qué mide CASEN sobre contacto sanitario

Se procesan las seis rondas: **2013, 2015, 2017, 2020, 2022 y 2024**, a nivel nacional, regional y comunal.
En 2013 (`s22a/s22b`, `s29a/s29c`) y 2015/2017 (`s19a/s19b`, `s26a/s26c`) las consultas generales y
controles tienen referencia de tres meses. Consultorio general o posta rural (códigos 1/2) permite una
aproximación específica a APS: personas con alguna de ambas prestaciones, sin duplicarlas.

En **2020** el cuestionario telefónico no incluye esas preguntas de prestación/establecimiento.
Se conserva por separado la consulta por enfermedad o accidente en tres meses (`s16/s17`), de cualquier
sector. **No reemplaza el indicador público ni el de APS.**

En **2022/2024**, `s20a_preg/s20b` mide medicina general en tres meses y `s26a/s26c` controles en doce meses.
Solo distingue público/privado/otro. Público no equivale a APS. No se mezclan controles de tres y doce meses.
Las preguntas identifican el último establecimiento declarado, no todos los establecimientos visitados.

El denominador es la población 15–64 con respuesta válida, no solo usuarios ni solo afiliados a FONASA.
Las no atenciones explícitas son cero; las respuestas desconocidas permanecen faltantes. Se informa su peso.
Estratos/UPM oficiales; `expr` para país/región y `expc` para comuna. El IC logit usa linealización y aproximación
normal; no se fuerza un IC cuando la proporción observada es cero o uno.
Cada ronda conserva su registro completo de advertencias por dominios con una sola UPM y una auditoría de
la unión geográfica. Se armonizan los códigos antiguos de Ñuble a la geografía INE actual mediante las etiquetas
oficiales; no se modifican estratos ni UPM. Los factores comunales oficiales se unen por folio + o hasta 2017,
y por folio + id_persona desde 2020. En 2015 solo 139 comunas tienen factor comunal oficial;
las demás quedan sin estimación comunal directa. Las estimaciones regionales incluyen toda la muestra.
Los factores comunales históricos tienen una base censal distinta del factor regional actualizado de 2013/2015.

Filtro exploratorio de precisión (decisión de este trabajo, no norma oficial): n válido ≥60, Kish ≥30,
≥5 casos y ≥5 no casos, ≥2 UPM, CV ≤30% y peso de faltantes ≤10%. Superarlo no valida una tasa comunal para el motor.


In [6]:
.t0 <- Sys.time()
if (rebuild) eca_health()
health <- read_result("casen_health_domains")
eca_show_table(health |> dplyr::group_by(anio_casen, geography, outcome) |>
  dplyr::summarise(cells = dplyr::n(), pass = sum(passes_screen), .groups = "drop"),
  "Celdas de salud disponibles y filtro exploratorio", digits = 0)
eca_show_table(health[health$geography == "national", c("anio_casen", "sexo", "ag", "outcome", "p", "se", "lo95", "hi95", "n_valid")], "Contacto observado: estimaciones nacionales por ronda", digits = 4)
eca_show_table(read_result("health_year_alignment"), "Alineación temporal del contacto sanitario")
eca_show_table(read_result("casen_health_indicator_availability"), "Indicadores disponibles en cada ronda CASEN")
contact <- read_result("insumo_contacto_comuna_anios_enpg")
stopifnot(nrow(contact) == 19376, setequal(unique(contact$anio), seq(2012, 2024, 2)),
  all(is.na(contact$p_aps_3m[contact$anio >= 2018])),
  all(is.na(contact$p_observed_commune[contact$anio < 2022])))
eca_show_table(contact[contact$comuna == 13101 & contact$sexo == 1 & contact$ag == "30-44",
  c("anio", "comuna", "p_observed_commune", "p_scenario_commune", "p_scenario_region", "source_a", "source_b", "weight_b")], "Contacto observado y escenarios: Santiago, hombres de 30–44 años", digits = 4)
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


anio_casen,geography,outcome,cells,pass
2013,comuna,aps_general_or_control_3m,2589,285
2013,comuna,public_control_3m,2589,225
2013,comuna,public_general_3m,2589,201
2013,national,aps_general_or_control_3m,8,8
2013,national,public_control_3m,8,8
2013,national,public_general_3m,8,8
2013,region,aps_general_or_control_3m,128,122
2013,region,public_control_3m,128,117
2013,region,public_general_3m,128,118
2015,comuna,aps_general_or_control_3m,1112,454


anio_casen,sexo,ag,outcome,p,se,lo95,hi95,n_valid
2013,1,15-29,public_general_3m,0.0370,0.0020,0.0333,0.0410,26085
2013,1,30-44,public_general_3m,0.0434,0.0028,0.0383,0.0492,18739
2013,1,45-59,public_general_3m,0.0881,0.0044,0.0798,0.0972,19399
2013,1,60-64,public_general_3m,0.1392,0.0110,0.1189,0.1623,4544
2013,2,15-29,public_general_3m,0.0681,0.0034,0.0617,0.0751,26496
2013,2,30-44,public_general_3m,0.0964,0.0049,0.0871,0.1065,21713
2013,2,45-59,public_general_3m,0.1459,0.0044,0.1374,0.1548,22568
2013,2,60-64,public_general_3m,0.1748,0.0088,0.1583,0.1928,5442
2013,1,15-29,public_control_3m,0.0248,0.0027,0.0200,0.0307,26194
2013,1,30-44,public_control_3m,0.0320,0.0024,0.0276,0.0371,18867


anio,a,b,wb,method
2012,2013,2013,0.0,Proxy CASEN 2013 (no CASEN 2012)
2014,2013,2015,0.5,Interpolation CASEN 2013/2015
2016,2015,2017,0.5,Interpolation CASEN 2015/2017
2018,2017,2022,0.2,Scenario interpolating 2017/2022; crosses pandemic
2020,2017,2022,0.6,Unobserved in CASEN 2020; scenario interpolating 2017/2022
2022,2022,2022,0.0,Observed CASEN 2022
2024,2024,2024,0.0,Observed CASEN 2024


anio_casen,outcome,available
2013,public_general_3m,TRUE
2015,public_general_3m,TRUE
2017,public_general_3m,TRUE
2020,public_general_3m,FALSE
2022,public_general_3m,TRUE
2024,public_general_3m,TRUE
2013,public_control_3m,TRUE
2015,public_control_3m,TRUE
2017,public_control_3m,TRUE
2020,public_control_3m,FALSE


anio,comuna,p_observed_commune,p_scenario_commune,p_scenario_region,source_a,source_b,weight_b
2012,13101,NA,0.0129,0.0343,2013,2013,0.0
2014,13101,NA,0.0156,0.0348,2013,2015,0.5
2016,13101,NA,0.0205,0.0338,2015,2017,0.5
2018,13101,NA,0.0258,0.0343,2017,2022,0.2
2020,13101,NA,0.0317,0.0384,2017,2022,0.6
2022,13101,0.0376,0.0376,0.0424,2022,2022,0.0
2024,13101,0.0511,0.0511,0.0548,2024,2024,0.0


Elapsed: 0.01 min


El insumo cubre **2012, 2014, 2016, 2018, 2020, 2022 y 2024**, con 2.768 filas por año.
Para contacto público general: 2012 usa 2013; 2014 combina 2013/2015; 2016 combina 2015/2017;
2018 y 2020 usan escenarios entre 2017/2022 (pesos de 2022: 20% y 60%). Estos dos escenarios
cruzan la pandemia y **no reconstruyen su efecto**. 2022/2024 sí tienen medición del mismo año.
`p_observed_*` solo contiene mediciones del mismo año; `p_scenario_*` también contiene proxies e
interpolaciones. El escenario regional aplica la misma tasa a comunas del mismo sexo–edad y región.
La interpolación comunal exige ambas celdas fuente; una ausente sigue ausente. No se fabrican IC temporales.
`p_aps_3m` solo tiene un escenario regional histórico para 2012/2014/2016; desde 2018 permanece NA.
Las tasas APS observadas de 2013/2015/2017 se conservan en la tabla por ronda, también a nivel comunal.
Para estimar APS actual se necesita una fuente que identifique el nivel de atención y personas únicas; los
recuentos REM de atenciones por sí solos no son probabilidades individuales. Contacto tampoco implica tamizaje,
intervención breve, aceptación o respuesta al tratamiento: son etapas diferentes del programa DIR.


## 6. Estimación preliminar: composición rural y categorías WHO

**Dos preguntas, dos resultados.** La comparación principal aplica directamente `peso` y `peso_nacional`
a las mismas respuestas válidas: muestra cuánto cambia ENPG con los pesos CASEN ya entregados. Incluye a
quienes tienen educación desconocida, con el tratamiento previamente documentado, y reproduce
35,946% → 36,025% para consumo del mes en 2024.
El cálculo rural posterior usa educación conocida y calibra cada grupo de respuestas válidas a su meta;
es una estandarización exploratoria distinta. Sus denominadores no se mezclan con la comparación principal.

Se muestran, en este orden, **nunca bebieron (lifetime abstainer)**, consumo en 30 días, categorías WHO y HED.
Nunca bebieron se deriva de `oh1 == 'No'`, nunca de ausencia de consumo en el mes. HED usa `db > 0`,
con el umbral del proyecto 5+/4+ por ocasión en 30 días; la referencia 5+/4+ no se presenta como un umbral
WHO universal. Un no consumidor del mes sin un reporte de episodios contradictorio tiene HED=0;
un consumidor con frecuencia desconocida, o una contradicción de estado/episodios, queda como NA.
Los indicadores no son categorías que puedan sumarse: HED puede coexistir con C1, C2 o C3+.

Se añaden **los siete años ENPG**, sin cambiar los pesos entregados anteriormente ni el motor.
Para cada sexo × edad × educación estimamos el consumo con ENPG y conservamos ese promedio al cambiar
la composición. Comparamos tres escenarios: ENPG original, composición CASEN urbana y composición CASEN nacional.
El contraste **urbano − nacional** aísla el componente composicional asociado a ruralidad:
positivo sugiere sobreestimación composicional; negativo, subestimación composicional.
La diferencia original − nacional incluye además otros desbalances del marco y de la muestra.

\[
\mu_U=\sum_x q_U(x)m_{ENPG}(x),\quad \mu_N=\sum_x q_N(x)m_{ENPG}(x),\quad B_{comp}=\mu_U-\mu_N.
\]

**No es una estimación del sesgo rural total.** CASEN urbana también incluye localidades urbanas pequeñas
fuera del marco ENPG. El consumo rural dentro de cada celda se supone igual al urbano; no se observa.
Con sexo, edad y educación fijos, este contraste es cero por construcción: por educación mostramos su
**aporte al contraste del grupo**, no un efecto rural dentro del nivel educativo.

Las metas nacionales son las ya verificadas. Se estima la fracción rural dentro de cada celda CASEN
(variable `zona` en 2013/2015, `area` desde 2017; 1 urbana, 2 rural) con los factores actualizados.
En años intermedios se interpola esa fracción según el calendario educativo; después se deriva
qU ∝ qN × (1 − fracción rural). Así se conserva exactamente la meta nacional. 2012 usa 2013.
Son estimaciones puntuales ponderadas; todavía no se propaga la varianza de las dos encuestas ni la interpolación.

Se aplica la [decisión de clasificación vigente del proyecto](decision_categorias_who_12g_2026-10-26.md):
NC = sin consumo en 30 días; C1: (0,20] g/día mujeres y (0,40] hombres; C2: (20,40] y (40,60];
C3+: >40 y >60, respectivamente. Los bordes exactos quedan en la categoría inferior.
Los cortes se documentan en WHO (2000), *International guide for monitoring alcohol consumption and related harm*,
[tabla 2.2.2, p. 54](https://www.who.int/docs/default-source/substance-use/who-msd-msb-00-4.pdf).
“Actualizada” se refiere a la regla vigente del proyecto, no a una nueva guía WHO de 2026; C1 no significa consumo seguro.

Dos escalas: **reportada, 12 g por trago**, y **anclada**, multiplicada por el factor APC anual ya usado por el
proyecto (`oms_factor_by_year.csv`, versión existente de junio de 2026; 2024 usa APC 2023).
El factor se mantiene fijo en los tres escenarios: recalibrarlo después escondería parte del cambio.
El volumen usa días, cantidad típica y episodios 5+/4+ de `ENPG_BINGE.RDS`, unido por año e ID original.
NC tiene volumen cero; un consumidor actual sin volumen válido permanece faltante.
Se excluyen educación desconocida y desenlace faltante; las metas se aplican entre respuestas válidas en cada celda,
lo que exige que esas respuestas representen a las faltantes dentro de la celda. Por ello el denominador puede
diferir de las tablas anteriores. NC/C1/C2/C3+ comparten denominador, pero “consumo actual” puede tener más respuestas válidas.


In [7]:
.t0 <- Sys.time()
source(here::here("__andres_control", "enpg_casen_rural_prelim.R"))
if (rebuild) eca_rural_preliminary()
eca_rural_tests()
rural <- read_result("rural_preliminary_estimates")
rural_cells <- read_result("rural_education_contributions")
stopifnot(nrow(rural) == 1680, nrow(rural_cells) == 2688,
  setequal(unique(rural$anio), seq(2012, 2024, 2)))
indicators <- read_result("enpg_indicators_by_group")
eca_show_table(indicators[indicators$anio == 2024 & indicators$scale == "anchored" & indicators$sexo == 0 & indicators$ag == "All" & indicators$outcome != "grams",
  c("outcome", "n", "original", "national", "change")],
  "Comparación principal 2024: pesos originales y nacionales, proporciones 0–1", digits = 4)
eca_show_table(read_result("rural_enpg_join_audit"), "Unión por año e ID y respuestas faltantes", digits = 0)
eca_show_table(read_result("rural_casen_dictionary"), "Variable urbano/rural comprobada en cada CASEN")
eca_show_table(rural[rural$scale == "anchored" & rural$outcome == "C3plus" & rural$sexo == 0 & rural$ag == "All",
 c("anio", "n", "original", "urban", "national", "rural_composition_gap", "unsupported_target_share")],
 "C3+ anclado: los siete años; proporciones 0–1", digits = 4)
eca_show_table(rural[rural$anio == 2024 & rural$scale == "anchored" & rural$outcome == "C3plus" & rural$sexo > 0 & rural$ag != "All",
 c("sexo", "ag", "n", "kish", "original", "urban", "national", "rural_composition_gap", "unsupported_target_share")],
 "Ejemplos 2024 por sexo y edad: C3+ anclado, proporciones 0–1", digits = 4)
eca_show_table(rural_cells[rural_cells$anio == 2024 & rural_cells$scale == "anchored" & rural_cells$outcome == "C3plus",
 c("sexo", "ag", "educ3", "n", "kish", "events", "missing_share", "m", "contribution", "screen")],
 "Educación: aporte al contraste, tamaño muestral y filtro exploratorio", digits = 4)
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


Elapsed: 0.00 min
Elapsed: 0.00 min
Elapsed: 0.00 min


outcome,n,original,national,change
lifetime,18204,0.2569,0.2572,0.0003
current,18092,0.3595,0.3603,0.0008
NC,17684,0.6556,0.6553,-0.0003
C1,17684,0.2379,0.2360,-0.0019
C2,17684,0.0512,0.0516,0.0003
C3plus,17684,0.0553,0.0571,0.0018
hed,17744,0.1708,0.1725,0.0017


anio,n_joined,n_education_missing,n_current_missing,n_current_volume_missing,n_lifetime_missing,n_hed_missing
2012,16322,109,82,312,32,346
2014,19326,508,131,848,52,888
2016,18354,230,197,485,93,661
2018,18734,217,342,757,79,895
2020,16114,105,137,563,18,664
2022,17023,107,108,383,31,458
2024,18231,132,139,408,27,487


anio,variable,label,codes
2013,zona,Zona,Urbano 1; Rural 2
2015,zona,Zona,Urbano 1; Rural 2
2017,area,Área,Urbano 1; Rural 2
2020,area,Área,Urbano 1; Rural 2
2022,area,Área,Urbano 1; Rural 2
2024,area,Área,Urbano 1; Rural 2


anio,n,original,urban,national,rural_composition_gap,unsupported_target_share
2012,15825,0.0491,0.0498,0.0498,0e+00,0.0583
2014,17912,0.0467,0.0486,0.0491,-4e-04,0.1143
2016,17503,0.0531,0.0536,0.0538,-2e-04,0.0000
2018,17442,0.0484,0.0507,0.0511,-5e-04,0.0137
2020,15326,0.0496,0.0511,0.0511,0e+00,0.0000
2022,16440,0.0547,0.0548,0.0548,0e+00,0.0000
2024,17569,0.0555,0.0576,0.0575,0e+00,0.0000


sexo,ag,n,kish,original,urban,national,rural_composition_gap,unsupported_target_share
1,15-29,1551,457.3090,0.0693,0.0681,0.0678,0.0002,0
1,30-44,2200,704.3430,0.0878,0.0901,0.0917,-0.0016,0
1,45-59,2029,669.3598,0.0730,0.0737,0.0741,-0.0004,0
1,60-65,1202,359.3864,0.0370,0.0369,0.0377,-0.0008,0
2,15-29,2041,596.8106,0.0528,0.0535,0.0533,0.0002,0
2,30-44,3251,908.3958,0.0457,0.0456,0.0455,0.0001,0
2,45-59,3051,995.5706,0.0309,0.0309,0.0303,0.0006,0
2,60-65,2244,658.7417,0.0111,0.0110,0.0106,0.0005,0


sexo,ag,educ3,n,kish,events,missing_share,m,contribution,screen
1,15-29,1,322,115.7969,7,0.0225,0.0373,-0.0002,TRUE
1,15-29,2,910,273.1340,57,0.0263,0.0773,0.0000,TRUE
1,15-29,3,319,74.5596,24,0.0328,0.0822,0.0004,TRUE
2,15-29,1,384,122.7287,6,0.0330,0.0070,0.0000,TRUE
2,15-29,2,1123,343.7305,34,0.0440,0.0673,0.0000,TRUE
2,15-29,3,534,132.5841,27,0.0397,0.0642,0.0003,TRUE
1,30-44,1,238,89.4837,32,0.0332,0.1711,-0.0025,TRUE
1,30-44,2,1106,364.2918,88,0.0235,0.0909,-0.0004,TRUE
1,30-44,3,856,267.8734,67,0.0314,0.0672,0.0013,TRUE
2,30-44,1,381,112.0521,19,0.0148,0.0308,-0.0003,TRUE


Elapsed: 0.00 min


### Ver el cambio y el soporte de los datos

El love plot muestra diferencias estandarizadas de las proporciones marginales frente a CASEN;
usa como denominador la desviación estándar binaria de la meta. La línea 0,10 es orientativa.
El ajuste exacto también verifica las 24 celdas conjuntas. Balancear estas variables no demuestra
intercambiabilidad ni elimina el problema de cobertura rural.

Para educación se informa n, Kish y eventos. El filtro exploratorio exige n ≥60, Kish ≥30,
≥5 casos y ≥5 no casos en indicadores binarios, y ≤10% del peso con desenlace faltante.
Kish solo resume dispersión de pesos: no incorpora conglomerados ni es una garantía de precisión.
La masa objetivo que no supera el filtro sigue visible; sus valores no se convierten en cero ni se ocultan.
No se ajustan interacciones adicionales por educación. Las figuras contienen puntos, no IC.


In [8]:
.t0 <- Sys.time()
eca_show_figure <- function(name, caption) {
  path <- file.path(eca_dir, "figures", paste0(name, ".png"))
  raw <- readBin(path, "raw", n = file.info(path)$size)
  tag <- htmltools::tags$figure(
    htmltools::tags$img(src = paste0("data:image/png;base64,", jsonlite::base64_enc(raw)),
      alt = caption, style = "max-width:100%;height:auto"), htmltools::tags$figcaption(caption))
  if (isTRUE(getOption("jupyter.in_kernel"))) {
    IRdisplay::display_html(as.character(tag)); invisible(tag)
  } else htmltools::browsable(tag)
}
eca_show_figure("loveplot_composition", "Balance antes y después en los siete años ENPG")
eca_show_figure("who_targets_2024", "C3+ en 2024: perfil original, urbano y nacional en dos escalas")
eca_show_figure("rural_composition_heatmap", "Dirección y magnitud del contraste composicional en todos los años")
eca_show_figure("education_contributions_2024", "Aporte educativo al contraste y celdas que superan el filtro")
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


<img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAABnIAAAXcCAIAAACZJIdQAAAABmJLR0QA/wD/AP+gvaeTAAAA
CXBIWXMAABcRAAAXEQHKJvM/AAAgAElEQVR4nOzdd2AU1fo38Gd3s5uQEBIgSCIl0iTSuSAI
EaVdAUGRqoCglCAiIEVA2o2ICJd2BWlJDEhJ6E1AQECQEoogHYIIJLSQAumb7fP+cX7Ou26d
2ZLNJt/PX3B25sxz5pw5O/NkdkbCcRwBAAAAAAAAAACAGFJPBwAAAAAAAAAAAOB9kFYDAAAA
AAAAAAAQDWk1AAAAAAAAAAAA0ZBWAwAAAAAAAAAAEA1pNQAAAAAAAAAAANGQVgMAAAAAAAAA
ABANaTUAAAAAAAAAAADRkFYDAAAAAAAAAAAQDWk1AAAAAAAAAAAA0ZBWAwAAAAAAAAAAEA1p
tVJFoVAoFIqEhARPB1KsHjx4wBqu0WgcruTmzZv79u1zePW0tDQWQ2FhocOVWOOSBpZkDx8+
HD58eFhYWPny5SMiIu7cuePBYJwcCcUpLy9vw4YNvXv3rlGjhq+vb3h4+Pvvv79jxw6B46RP
nz5sXF2/ft3uwgaD4fjx459//nmjRo0CAgIqVarUtWvX+fPn37t3z+LykZGRCmHUarXxijVq
1FAoFHXq1NFqtdaCyc3NZevev39fSEs9LisriwWcm5vr6VhMlfrpBQAAAADArZBWK1W0Wq1W
qzUYDJ4OpFhxHMca7nANixcvbtiwYWpqqjNhOBmDDc43sCTLzMx8+eWX16xZ8/Tp08LCwtu3
b1eqVMlTwbhkJBQDvV7/9ddfBwUFDRkyZNeuXY8ePdJoNA8ePNi6dWvfvn1DQkJ+/fVX2zXk
5eXt3LmTjatly5bZXjgjI6Nhw4YdOnRYtmzZjRs3lEpldnb2oUOHpk2bVqdOnY8++sg8I6PT
6bTCWFzx3r173377rY2Q2Locx9mOvOQosYdw6Z5eAAAAAADcDWk1AEpKSvJ0CGXXnj17VCpV
QEBASkqKwWDQarWVK1f2VDBeMRLUanW7du2io6OJKDo6+v79+xqNxmAwFBUVnTt3rnXr1vn5
+Z06ddq9e7eNStin/fv3J6K4uDilUmltyfz8/Pr16ycnJwcEBMybN+/MmTN37969ffv2oUOH
evfuTUTr16/v3r27xQxXv379UuxRKBQWt/vVV1/99ddfYnYMAAAAAABAcfPxdAAAUKY9ePCA
iPr06RMeHk5EPj6YlOzo37//mTNnAgMDL1++XLt2bb7cz8+vVatWSUlJ77///vbt2/v06fPo
0aOwsDCLlcybN4+Ivvjii3v37l24cGH37t0DBw60uOSSJUtycnJCQkL+/PPPihUr8uUvv/zy
W2+9tXnz5gEDBhw5cuTIkSP//ve/TdatUaMG61bHvPfee1evXpVKvf7PPyEhIV50Yx0AAAAA
AAjn9ZcrAODVWLohKCjI04F4h8OHD//0009EdOzYMeOcGk8qla5bt87f399gMMyYMcNiJffv
309OTiaiJk2ajB49moi+/vpra1tcvXo1ES1YsMA4p8b74IMP2rdvzy/mKpGRkUR048aNlStX
urBaAAAAAAAA10Jaray7cePG2LFj69WrV65cuVdeeWXKlCkpKSkmy/z0008KhSIkJMTiU9tO
nDhh/mlqaurUqVNfeeWVcuXKRURETJkyRfjzqtLT0xUKRY0aNYjozz///Oijj0JCQipVqtSn
T59Tp065tnUJCQkKhWLnzp1ENGbMGIVCYTF3IEpqaurw4cOrVKkSEhIycOBAazFzHHf8+PFP
P/2U7aWQkJCuXbvGxsaaPMHdBoE1sP1Zp04d9u+pU6fWq1cvICCgbdu2K1eutPac8tu3b8+c
OTMiIoLtupkzZ2ZkZFhrr2N9PWzYMIVC8c033xDR999/z56bfvToUSJ6/PixQqFo2rSpwWCY
N29eWFhYjRo1pkyZYvxbRSHbFd52ISPBmVHNON/pkyZNIqK33367RYsW1pbx9/efM2fOO++8
06ZNG4sLrFu3jojeeustX1/fd955h4hu3759+/ZtiwE/ffqUiGz8Mnf48OEvvPBCenq6wCYI
MXr0aJbvGzt27JMnTxyux7HBL7ab7B4sNl5ZIGQGdqAVzo80AAAAAAAQhINShPXp+vXrhSys
1+tHjhxpcVTMnTvXYDDwSxYUFLDya9eumdczePBgIpoxYwZfsnDhQovVfvfdd0ICY5fxoaGh
x44dM69k6NCher3eeHn+KlStVott3fr1600+DQgI4CuJi4uTy+VyubygoMB2zPyV//fff2++
xREjRpjEXFBQ8MYbb1gMLzQ0NDMz024DhdfA9mfNmjVPnDhh/nu6unXrmrTOYDBMnDjRvFqp
VPrrr7+aNNyZvmYjx8Thw4c5jnv48CER1atXLyoqiv8oJCSE340Ctyu87bZHgpMtFdtl1rDm
ENGBAwcEbtScTqcLDAwkop9++omVtG7dmojGjBljcfnmzZsTUWRkpMkYtq1Vq1ZENHHiRLHh
hYaGElFiYmJRURHL5bVt29Z4OuI4Licnh+2Hu3fv2q5N7ODnRHaTwIMlMzOTlefk5PCFwmdg
sa1wfnoBAAAAAACBkFYrVdjVkcC02rhx49jyK1asyMvLMxgM2dnZ/G/BFi9ebLxw3759iWj6
9OkmlajVanaZl5yczEpWrFjBavjuu+9ycnIMBkN+fn5sbCwr3LBhg93A2DWkTCaTyWR169a9
evWqXq9XKpV8zYsWLTJe3uJloajW9erVi4iWLVtmEklMTAxbXnhajYiaNWuWnJzMYl61ahUr
XLBggfHyXbp0IaLw8PDz58+rVCqDwZCXl/fjjz+yhYcMGWK3gcJrYPtTLpdLpdLOnTvfvXtX
r9erVKpt27axhb/88kvjzS1YsIDfz6wH09PThw4dSkRSqfTp06f8ks73Ncdx06dPJ6LRo0cb
F7K0GrNlyxatVltYWHjv3j2x2xXbdmsjwSUtFdXpFh06dIgt/PjxYyFbtOj06dOsksLCQlay
ZcsWdsQVFRWZL79582a2fPPmzX/++Wd+LducT6txHMcn1hMSEoyXEZtWEz4AOJHdJPBgsZhW
Ez5HiW2F89MLAAAAAAAIhLRaqcKujoSk1fgffP3yyy8mH8XHx7OP0tPT+UJ2fRscHGxy2wgr
r1atGivPzs5mWbZdu3aZVPvLL78QkUKhsHtZzt+S88ILL+Tn5xt/tHHjRvZRbm4uX2h+WSi2
deyFhsuXLzdZ2IG0Wq1atVQqlfFH7DYomUzGt4U9pJ+Ibt++bVLPokWL2AW58X42b6CoGvj9
2bx5c5MbjlhSwM/Pj184OzvbYiLDYDA0adKEiMaPH88v6Xxfcxw3c+ZMIho7dqxxIZ9WGzVq
lMnyorYrqu2clZHgkpaK7XSL+OyeRqOxvaQNH3zwARH179+fL8nLy2PVbtu2zXx5g8HAJ4CY
li1bzps379y5cyZD3ZhL0mp8tAqF4tmzZ/wyYtNqwgeAqG4SfrCYp9VEzVGiWuH89AIAAAAA
AMLh2WplFEsYtWnTxvzlfUOHDq1ZsyYR8TksIoqMjJTL5Tk5Obdu3TJemF0BTpw4USKRENH2
7dsNBkP9+vXfe+89k2o7d+4cGhqq0Wj4O27sWrFiRfny5Y1LBgwYwK669+3b58LWWTNy5Eh2
nAQEBAiMeeXKlb6+vsYlgwYNqly5sl6v379/Pyt58OBBmzZt2rdv//LLL5us/tZbbxGRwWCw
9tQnZ2r45ptvTH5B1q1bNyJSqVT8E5d+/vlnIgoNDWXpDJ5EIlm8eHGDBg10Oh0rcW1fWzNk
yBCTEse2K6Tt1rikpc53OhHxT+ySy+W2l7SmsLCQ3X326aef8oWBgYHsCWsWX1wgkUiWLl36
888/161bl5VcuHBh2rRprVu39v

<img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAABnIAAAOECAIAAAAaKTjwAAAABmJLR0QA/wD/AP+gvaeTAAAA
CXBIWXMAABcRAAAXEQHKJvM/AAAgAElEQVR4nOzdeWAM9/8/8Fd2sxGJHCqUiDuIuOuIo3oQ
tEKDuIL4uI+2KErcV+sojY+jStx1n20JLeou0jiKqKLOkIggcm822d2Z3x/v72d+09nN7sxm
kwjPx1/Z97znPe95v98zmX3tzHsceJ4nAAAAAAAAAAAAUEJV1BUAAAAAAAAAAAAofhBWAwAA
AAAAAAAAUAxhNQAAAAAAAAAAAMUQVgMAAAAAAAAAAFAMYTUAAAAAAAAAAADFEFYDAAAAAAAA
AABQDGE1AAAAAAAAAAAAxRBWAwAAAAAAAAAAUAxhNQAAAAAAAAAAAMUQVgMAAAAAAAAAAFAM
YbXXSnp6+pYtW7p3716pUqUSJUpUqVKld+/e+/bty83NzWsVnuePHj0aEhLy9ttvlypVqn37
9uvWrdPpdIVZ7cJx6dKlESNG1KxZs2TJklWrVh02bNjly5fzypyUlBQeHs4yv/POO998801G
Roa9ChdLTk4uVaqUk5OThT6S49GjR05OTvkvB+wlMTGR9UhWVpbVzC9evGCZ09LSCqFuecEo
Muvu3busWYxGY1HXBQAAAAAAXi0OPM8XdR3ADoxG47x582bNmmV2qZub288//9y2bVtJek5O
Trt27c6dOydJL1269OXLl6tVq1YgdS10BoOhb9++e/bsMV00aNCgdevWqVT/ii+fOnWqXbt2
HMeJE93c3K5evVq9evV8Fi7G83y7du1OnjxJRDk5OU5OTvJ3SiIuLq5q1ar5LwfsJTEx0dvb
m4gyMzNdXV0tZ37x4kXZsmWJKDU11cPDozDqZw5GkVl37typVasWERkMBrVaXdTVAQAAAACA
VwjuVnsd5OTktGnThsXUZs2a9eDBg9zcXI7jsrOzY2JiAgICMjIy2rVr9/PPP0tWDA0NPXfu
nKen55EjR7Kzs41G47179z766KOUlJSGDRtmZmYWxd7Y3+TJk/fs2aNWqzdv3pyZmclxXHp6
+nfffUdEGzdunDt3rjhzQkICi6mNHz8+NTWV47jHjx936NAhIyOjSZMmWq02P4VLbN26lcXU
AAAAAAAAAKDYwd1qr4Pg4OADBw7kdTsVx3G9e/feu3evSqWKj4+vUKECSxduwbh06VKTJk2E
/AaDoXz58snJyZGRkcOHDy+0vSggwn1Av/32W2BgoHjRtm3b+vfvT/++Rahr16779+8PCwvb
vHmzkNNoNDZr1uzKlSsLFy4MDw+3uXCxpKSkihUrCo+V4W6114yiu9VeERhFZuFuNQAAAAAA
yAvuViv2fvvttwMHDhDRyZMnTWNqRKRSqX744QcXFxeO46ZNmyakp6WlffLJJ61atRLH1IjI
0dHx008/JSJWrFnCvFF2240Cc+zYMSKqWLGiJOxFRKGhoRqNhoj++OMPlpKSkrJ//34imj17
tjinWq1esmQJEc2fP1/8cKiiwsV4nu/WrZvRaJRsCAAAAAAAAACKC4TVir0JEyYQUadOnSTR
MTEXF5evvvqqS5cuLVu2FBKbNm26f/9+04nViEiv1xORo6Ojhe3q9XqWzTZxcXHh4eF16tQp
WbKkn5/fpEmT4uLiJHmSkpKcnJxq1KhBoncIuLq6tmrV6vvvv5c5q3pmZqa/v//HH39sukil
UtWsWZOI4uPjWcqFCxeIyNXV1TRA2aJFCyJKT0+/d++ebYWLrV27Njo6+vPPPx84cKCcvciP
GzdujB49mr1OoU6dOpMmTXr48KGiEnieP3LkSK9evTw8PCpVqhQeHp6SkmI0Gllo1WAwsGwW
JrwXJn0XMgvkjASrbt++PX36dD8/P7aP06dPf/bsmdkdOXXq1KhRo9jmvLy8PvroozVr1uTk
5IizsYFXp04dInr8+PHYsWN9fHzc3d2Dg4OPHz/O8uTm5q5cubJx48bspRbr16+3MJ99XFzc
kCFDypYt6+Xl1bdv37Nnz0oy5PXKApkVtswuhYglJCQ4OTk1bNiQ47gFCxZUqFChUqVKkyZN
kjwiLf8Yr1SpEhH9888///nPf7y8vN56662QkBDTVhLIHNI2nEB4nj98+HDPnj09PDz8/Py+
//57y68pkLOPcppLZh/JbHkAAAAAACg8PBRnT58+Zf3466+/2qtMrVZbunRpItq2bVteeZ48
eZKf8bN48WKzo3Hp0qXibGzvKleufObMGdOJ/319fTMzM22rAGM0GtkNZYcOHWIp7Ja0Ll26
mM1fvnx5Ivrpp59sK1zw6NEjIqpQoYJOpxPCATk5OZJsa9eu1Wg0Go1Gzm6aLcdoNOb1GO+8
efM4jpOzIzqd7oMPPpCs7urqeu3aNfa3Xq+3UAfmn3/+kWRmZI4EC9gseKYlqFSqEydOiHNm
Zma+9957ZjdXvnz558+fCznZwPP19T169Khp5rVr16akpPj6+krSJcNGOEZWrFhhWsjQoUON
RqOQ+fnz5yw9NTXVhgpboKgQCz0o9vjxYyKqWbPmsGHDhNK8vLzEe6ToGC9fvrzZSQYHDRok
LpNXOKSVnkDMDvXAwMArV66wvw0Ggzi/zH202lzy+0hOywMAAAAAQGFCWK14O3LkCPtmlZCQ
kP/StFrtyZMn/fz8iKhJkyaSCIhYfsJqK1euFL58sncCZGRkrFmzhiVu2bJFyMm+FWs0GpVK
FRgYeO/ePaPRqNPphNduTp482Zb9/B/2CCcRPX36lKWwL6sTJkwwm5995V6xYoVthTNGo7Fx
48ZEdPHiRd5iICMyMpItsjmsNmbMGJa4cuXK9PR0juNSUlKEtyhERETI2ZG+ffsSkZOTU1RU
VE5Ojl6vP3/+fJkyZdzd3Vk5NofV5I8ECxYtWsTyf/vtt6yQpKSkQYMGEZFKpRI3fseOHYmo
SpUqFy5c0Ol07OUSmzZtYqsPGDBAyCkMPCLq0aPH48ePWdN1796d/ncfop+f3/Xr1w0Gg06n
i4iIYIX8+eefQiHCMUJEjRo1unXrltFo1Gq1q1atYomLFi0SMpsNq8mvsAWKClEUVmN27dql
1+uzsrLu378vZFB6jKvVarVa7evrGxsby1pJKOHbb78Vb1rRkFZ6AunVqxcRubi4HDlyJDc3
12AwXLhwoWLFivXr12f5xWE1+ftotbnk95HVogAAAAAAoJAhrFa8CV/tcnNz81mUi4uL8IVt
6NChlgu0OayWkpLCbhsxveeL3Rnk5OSUlZXFUoR78Ro3biy5HYMFU5ydnWXecmVKp9P5+Piw
uImQyAJnixcvNrtKaGgoEU2aNMm2whl2Q9wXX3zBPhZcWO327dss5ejRo5LM69evZ4uSkpIs
F3vz5k2W88KFC+L0xMREFnUiW8NqikZCXlJSUlixkjsrOY5r0KCBuJ3ZHYJEdPv2bUkh3377
LRGpVCphLOU18IQHSzUaTVpamriQpk2bEtGCBQuEFOEYqVatmk6nE2dmb8NQq9UZGRksxTSs
pqjCeVFaiNKw2siRI02X2naMlytXTmgNZuvWrWyR0NRKh7SiE4gw1GNjY8U5k5OThXOjEFZT
tI+Wm0tRH1kuCgAAAAAACh/mVivexN/z81NOTk6OeHaezZs3R0REiOfm37dvn5MImwuJiJz+
TfiKmJe9e/dyHFe7du2uXbtKFgUGBpYvXz43N1e4BU/w9ddfS57hYtOZ6XQ62+aHMhqNPXv2
jI+Pd3FxWbt2rZCemZlJRKVKlTK7Fnufo9VpjPIqnIju378/fvz4ihUrfvPNN1YrOXz4cHaU
2vYeSRaVa9myZfv27SWLBg0aVLlyZSISIhd52b59OyukWbNm4vTy5cvPmDHDhloJbBsJEr/8
8gurTJ8+fcTpDg4OERER/v7+4n

<img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAABnIAAAOECAIAAAAaKTjwAAAABmJLR0QA/wD/AP+gvaeTAAAA
CXBIWXMAABcRAAAXEQHKJvM/AAAgAElEQVR4nOzdd1wT5x8H8G8SExkCKrhRBEERB+6t1UrV
OuusWvWnVawTrdZVcNdRLdY9UHHPOlq1dYtaR11VUXFUwY2IiGwk435/PPUak5DchYT5eb/8
Qy7PPXme5557kuebu+ckHMcRAAAAAAAAAAAAiCHN6QIAAAAAAAAAAADkPQirAQAAAAAAAAAA
iIawGgAAAAAAAAAAgGgIqwEAAAAAAAAAAIiGsBoAAAAAAAAAAIBoCKsBAAAAAAAAAACIhrAa
AAAAAAAAAACAaAirAQAAAAAAAAAAiIawGgAAAAAAAAAAgGgIqwEAAAAAAAAAAIiGsFq+kpiY
uGXLlm7dupUvX75w4cJubm5ffvnl3r17MzIyMtuF47hjx4517969VKlSRYoU+eyzz9atW5ee
np6dxc5mT58+VSgUCoXCSLOY7eHDhyxztVpt8cxzs4iIiEOHDuV0KYhyU0kKlDdv3rCen5CQ
kD07ZgOrjhWZiY6OZm+akpKSbW8KJqGjAgAAAIBBCKvlE2q1etasWU5OTgMGDNi/f//z588z
MjKePn26e/fuHj16uLi4nDp1Sn+v9+/fN2/evG3btvv27Xv9+nVKSsqJEyf8/f3Lli0bFRWV
/bXIHhzHKZVKpVKZ5zLPtYKDg6tVq/bkyZOcLkguKkkBZHbPz7WnTE6dzrm2QQq4XHtcCubn
DgAAAEAugbBafsCiY9OnTyei6dOnR0VFZWRkaDSatLS0S5cuNWzYMCkpqXXr1r/++qvOjn36
9Dl//nzRokWPHj2alpamVqsfPXrUrl27+Ph4X1/f5OTknKgN5D0XLlzI6SL8K/eUBAAAAAAA
API9hNXyg169el28eNHBweHRo0czZsyoWLGiXC6XSCQ2NjYNGjS4cOFCjx49iKh79+7R0dH8
Xv/888/+/fuJ6MSJE23atLGxsZFKpR4eHgcPHnR2dk5KStq+fXuOVQkA8hQXFxeO4ziOc3Jy
yp4dAbITOioAAAAAGISwWp53/PjxAwcOEFFYWJiHh4d+AqlUumnTJjs7O41GExgYyG9PSEjo
3LlzkyZN6tatq52+UKFCI0aMICKWrUH86j8WqwYAAAAAAAAAQJ6CsFqeN378eCJq3769TnRM
m52d3ezZszt16tS4cWN+Y7169X777bfz58/rp2dLtBQqVMjI+2ZlJReO406fPj18+PCqVava
2tq6uLi0a9cuJCTk/fv32sliYmIUCkWlSpXY/ydNmuTl5WVvb9+kSZOVK1dmtjbz/fv3g4KC
vL29bW1tq1atGhQU9Pr1awuWSjv9kSNHevbs6eTk5O3tvXLlSuOPKbhz587o0aO9vLxYwSZO
nPj48WMhBTNJeJUFlkF4y2/btk2hUOzbt4+IRo0apVAoihUrRkQvXrxQKBS+vr4ajWbevHll
ypQpX778xIkTU1NT+X1FNfi1a9f8/f0rVqxoa2vr5eU1aNCgkydPchxnsiS8J0+eTJo0ib2X
t7f3xIkTzViCzbJNzdq5atWqRPTs2bMxY8a4uro6Ojp26dLl5MmTLE1GRsaKFStq165ta2tb
p06d9evX63Qzlkn58uWJ6MGDB//73/9cXFyKFy/evXv3c+fOZVYRUb3RZOOT0QXdjTea8ZXg
hTejqIFC7MlukpE14/nHmKhUKrZFyNnBPHnyZPDgwSVKlHBxcenbt29mB9SqI6r1Bi7hsucQ
F4SOCgAAAACWx0Fe9urVK3YcDx8+bKk8U1NTWTxi27ZtmaV5+fKl2f0nOTm5RYsWBntj6dKl
Y2Nj+ZSsdhUqVDh79qxUqhsC9vT0TE5O1s5Zo9GMGzdOP1upVHrq1Ck+GT/bef/+vRmlYtLT
01u2bKmT0s/P7/r16+z/KpWKT6xWq4cOHWow8zlz5mg0Gu2c165dK5fL5XK5Tu0MElhlsWUQ
3vKbN2/WedXe3p7juGfPnhGRl5eXv78//5KLi4tarTajwQ3WkYhatmyZkZFhvCTMwoULDeaw
ePFik41svaZm7ezp6Xns2DH9xGvXro2Pj/f09NTZ3qlTJ/2DVbp06bCwMP1MBg0axLe5GT1B
YONzHBcbG8u2v3v3TlSjGdzRjGYUPlAI73sGxwqDjKR88OABe0mpVLItxs8OfnRdtmyZfgmH
DBmic0CtN6KK7SrWY71DzBScjgoAAAAAFoewWt529OhR9mX6xYsXWc8tNTU1LCzM29ubiOrW
rctPAvVlJazWtm1bInJzc7t8+XJ6erpGo0lMTNy4cSPLcMCAAXxKNgmRy+VSqdTPz+/Ro0dq
tTo9Pf2XX35hiSdPnqyd84IFC9j2n3766d27dxqNJiYmZtCgQWx29OrVK5bM4AxEeKmYXr16
EZGdnd3Ro0czMjJUKtXly5fLlStXo0YNtot2WC0gIIBtXLFiRWJiokajiY+PnzVrFtsYHBys
nfOaNWvYdiFhNYFVFlsGsS3ftWtXIlq6dCm/hQUOmF27dimVypSUlMjISDMa/MyZM0SkUCiO
Hj2akpKi0WhSU1MPHjwok8mIaO3atcZLwnHcihUrWLaLFy9mrZSUlBQSEsI2btmyxWQ7W6mp
+XYmoh49ejx79owl7tatG8vWy8vL29v71q1bKpUqPT09ODiYZfL333/rZCKTyWQymaenZ3h4
uFqtTk1N5Wv9008/aVdEVE8Q3vgGgw5CGi2zaIUZzSiwuwrve1YNq2V2dvCjKxHVqlXr3r17
7ICuWrWKbVywYIF51RHbUKK6ilVZ7xAzBaejAgAAAIDFIayWt/EzZ+3LRsxjZ2fHz+WGDBli
PEOzw2pPnz5lO96/f1/npZ9++onNYfif9/lr8WrXrq1zgQabBdnY2PCJ4+PjWWKdi+w0Gk3N
mjWJaOzYsWyL/gxEVKk4jrt79y5LHx4erp04Li6Ob0Y+rHb//n225dixYzqZr1+/nr0UExPD
bxQeVhNeZbFlENXyHMexMNDy5cv5LXzgYNiwYfolF9XgbKW/+fPn66RctmyZt7d3YGCg9kb9
ksTHx7MLQ/bv36+TA7tGTKFQpKSk6BdSm5WaOrN25m89k8vlCQkJ2pnUq1ePiObNm8dv4TMp
WbJkUlKSduKtW7eyl/hMxPYE4Y2vH3QQ2GgGoxUWaUbOUHcV1fesHVYzeHbwo6u7u3t6err2
S+ySTJlMxh9o642oYruKVVnvEHMFrKMCAAAAgMUhrJa3TZ8+nX2ZzmI+6enppEWhUMybN0/7
e/+ePXvkWti1KkQk/9iTJ0+Mv9G5c+caN27csmVL/ZfCw8NZnvxMkp+E/P777zqJb926xV5K
S0tjW7Zt20ZEpUuX1pmucBx3/PhxHx+fUaNGsT/1ZyCiSsVxHHvyg5+fn376+fPns/R8WI3d
W9S4cWP9xBqNpkKFCmTudR/Cqyy2DKJanjMaVrtw4YL+m4pq8IEDBxLR0KFDjTSFkZKsXbuW
iKpUqWKw7qVLlyaiffv2Gc/WSk1tpJ3t7e2JaMSIETrbJ06cSERff/01v4XP5JdfftFJrFar
WQX5eIHYniC88fWDDgIbzWC0wlLNqN9dRfU9a4fVDJ4dfFhN/9Z+tVrt7OxMRDt37jSjOqIa
yiIDl84HR2ZMfnBY7xBzBayjAgAAAIDF4ZEFeRsf3soiuVzO7vBKTExcvnx5RkbGlClT+vTp
wyfQaDRKLfyi6cqPcR+vYq6vadOmFy5cMLgIlIODA/9eOi/xd1by2NySiPhVwP/8808i6tOn
j/6aNX5+fnfu3DG4UJF5pWL33vbt21c/fadOnXS27N+/n4i0F1HiSSSSIUOGENGvv/6aWdmM
EF5ls8sgpOWNq1ixov5GUQ3eoU

<img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAABnIAAAOECAIAAAAaKTjwAAAABmJLR0QA/wD/AP+gvaeTAAAA
CXBIWXMAABcRAAAXEQHKJvM/AAAgAElEQVR4nOzdd3xT1f8/8HeSJnQPqAqlDJkFKUM2OBhl
g3zYG0RlKfQjqIACCshywIchQoECliHzIwiytYqyQQRlKVtWC6UjbZpm3Pv94/y8v/tJ0vTe
m6QJ6ev5Bw96c3PuOeee+86979ycq+J5ngAAAAAAAAAAAEAOtbcrAAAAAAAAAAAA8ORBWg0A
AAAAAAAAAEA2pNUAAAAAAAAAAABkQ1oNAAAAAAAAAABANqTVAAAAAAAAAAAAZENaDQAAAAAA
AAAAQDak1QAAAAAAAAAAAGRDWg0AAAAAAAAAAEA2pNUAAAAAAAAAAABkQ1oNAAAAAAAAAABA
tpKeVsvKykpOTn7llVdiYmJKlSpVpUqVQYMG7dixo6CgwENbvHjxok6n0+l0s2fP9tAmCnP1
6lW2aavVWsybdotHjx6x+mdnZxe2zpUrV9g606dPL8aq+aiLFy/u3r3bu3W4ffs22yMmk8m7
NZHOF/qNKeaaFE+IyMnJWbduXc+ePStUqFCqVKlKlSr169dv+/btTkYIz/MHDhzo1avXM888
Exoa2q5du1WrVhmNRs9V0hd47th50j8L7J0+fXrUqFHVq1cPCgqqXLnyiBEjzpw542T9tLS0
SZMmsfWff/75Tz75RK/Xu6twQUZGRmho6JMV/QAAAABANr6kMpvNkydPLqxbAgMDU1JSOI5z
+3b79u1LRA0bNrRarW4v3Lk///yTtc5isRTzpt3i4cOHrP5ZWVmFrTN48GAiql+//hPaRjf6
/PPPieiLL77wbjVu3rzJ9lpBQYF3ayKRj/SbV2ri6RBhsVhmzJhRWNQNCwv7/vvv7d9lNBpb
tmxpv35UVNT169c9UU8f4blj50n/LBAzm819+vRxOKKGDx/u8HM2NTVVrbb9TjEsLOzatWuu
Fy7gOK5169ZPVvQDAAAAAAVK6N1q+fn5jRs3njdvHhG9/fbbV65cKSgo4DguPz//1KlTXbp0
MRqNQ4cOffXVVzmOc+N2Hz58uGXLFp1Ot2fPHvvTenBRRkbG+vXrtVrt3r17NRqNt6vjZUeP
HvV2FZ5IvtNvvlMTtygoKHjxxRc/+ugjIvroo49u3LhhMplY1D1x4kTTpk31en3btm137Nhh
88YBAwYcOXIkMjJy//79+fn5Vqv12rVrHTt2zMzMrFevXm5urjdaA75i8uTJW7du1Wg0KSkp
ubm5HMfl5OR88cUXRLRmzZqZM2farH/37t22bdtyHDdhwoSsrCyO4/7+++/27dvr9fqGDRsa
DAZXChdbv359amqqu5sLAAAAAL7H23k9L+A4rlWrVkQUHh5+4cIFh+ts2rSJ9c+kSZPcuOlp
06YR0e7du91YpnT+dIeCQ+wiZ8eOHd6uiE/o2bMn+cBdV0/c3Wo+0m9eqYlHQ8Qrr7xChdwT
xPO81Wrt3bs3EanV6nv37tlX6fTp0+L1zWZzmTJliCgpKcntVfURuFutSMItzAcPHrR5af36
9ewlm7ubu3fvTkRDhgwRL7RYLA0aNCCiefPmuVK44MGDB+Kvdp6U6AcAAAAACpTEG6Y2bdr0
448/EtGJEydq167tcJ1+/fqxb6Q/+eSTixcvumvTM2fO5Hm+S5cu7ioQxKZNm8b/c9UEAL7j
4MGD3377LRGlpqZWqVLFfgW1Wv3VV18FBwdzHDdlyhRheXZ29iuvvNKiRYuGDRuK1w8ICHjz
zTeJiBXr0P3799kMYm5rBviYQ4cOEVH58uUTEhJsXhowYIBWqyWi48ePCwszMzN37txJRDaT
b2o0mgULFhDRnDlzhFvU5RYu4Hm+R48eVqsVU3wCAAAAlAQlLq3GcVxiYiIRvfnmm3FxcU7W
HDNmzLPPPktEEydOFBY6mUNamATaYrHYl3br1q1JkybVqlUrKCgoLi5u4sSJt27dklt56YXw
PL9v374+ffpERETExcV9+eWXDqemVtycK1euTJ06NS4uLigoqFatWlOnTk1PT7evw48//jhm
zBhW4ejo6I4dO65YscLJ4yCcF+vkkQUXLlwYN24cm1K6Vq1aEydOFG70EEtLS9PpdFWrViXR
lNUhISEtWrT48ssvZc0qLbF1bIsVKlQgoj///HPYsGHR0dGlS5fu1avXL7/8UljhEptz9+5d
nU5Xr149juPmzp1brly5ChUqTJw4cc2aNTqd7r///S8RjR07VqfTRUVFya25W5qvgJShxUjp
JVl7fMOGDQ77rbB+Fn4vJrc3zpw5M2LEiMqVKwcFBVWvXn348OFsTrEiayJwPZ5IDBHu2uI7
77xDRJ07d7bJjokFBwd//PHH3bp1a968ubCwUaNGO3fuPHLkiP36ZrOZiAICApxs12w2s9UU
kHWMK4gq0oe6sooJK8va0RKDj3NSRotbonFubm7t2rU7depk/5Jara5evToR3blzR1h48uRJ
IgoJCbHP7TZr1oyIcnJyrl27pqxwwcqVK48dOzZ27NhXX31VSisAAAAA4MnmjVvkvOn8+fOs
4ZcuXSpy5VWrVrGV9Xo9W+LkVznCz2rMZrPNS5999pnDzl+4cKH0mksvxGg0sl+5iiUkJJw9
e5b9X/jhj4LmsClp7KuhVqt/+OEHYbXc3NyXXnrJYYXLli378OFDm81JKdbhIwusVuvIkSMd
bmj27Nk2D5148OABEVWsWPHw4cP2c9tVq1YtNzdXyr6Q3jq2xbJlyzqcZMd+0mtZzfn777+J
qHr16iNGjBBWi46O/uqrr2zeGxISomy/uNh8Xs4P2SQOLVm9JGuPp6SkOOy3wvqZ7Tu5Xeqw
jUTUqlUrk8nkvCaM6/FEeohwyxbZXiCivXv3SqxhkQwGA0s1btiwobB17t27x7aroHy5x7is
qCJxqDs8dmSNN1k7WlbwcULiaHFXNC6M1WplN5R99913wkJ2S1q3bt0cvqVs2bJE9M033ygr
nLl9+zYRlStXzmg0PnE/gQcAAAAABUpcWm3x4sXsNFfKgzgvX77MVj569ChboiAPtXTpUuGi
gk2QrNfrV6xYwRauW7dOSrVlFcIeNhocHLx//36TyWSxWE6ePFm+fPn4+HibSykFzfn000/Z
8s8//5zVJC0tbfjw4USkVqsfPHjAVuvQoQMRVapU6eTJk0ajkc30vHbtWvbeoUOH2mxOSrEO
02rs3kMiWrp0aU5ODsdxmZmZwkzS8+fPF2+FXchptVq1Wp2QkHDt2jWr1Wo0Grdu3crWnzx5
spTdIb11bIsajUaj0VSrVu38+fNWq9VgMAg79PPPPxeXLKs5LN3DbN682Ww25+XlCc9G7NGj
BxEtXrxYWc3d0nxeTlpN4tCS1UsK9rh9vznvZ1m98dNPPxGRTqfbv39/Xl4ex3EGg2HXrl1s
GqaVK1c6rwnvpngiPUS4ZYv79+9nK9+9e1dK9ZwzGAypqansXuOGDRvaf40hcCWtJvcYlzXG
JA51h8eOrPEma0fLCj6FkT5a3BWNC8N+wklE4tDBMuPvvPOOw7ewFOSSJUuUFc7zvNVqZXO0
nTp1in8CZ5YEAAAAAAVKXFqN/SijevXqUlYWfuS1evVqtkRuHiozM5N9D2//BfiBAwfYBXZe
Xp7zasgq5NKlS6wa58+fF6+ZkZERHBxscymloDlsoc0dIhzH1a1bl4jefvtt/p+v64noypUr
NsV+/vnn7NJRfO+DxGLt02pXrlxhSw4cOGCzoeTkZPZSWlqasFC4a6ZBgwY2eVV2oRsYGFjk
TRmyWids8emnnxbueWSESa+zs7OVNUdI94wePdq+nvYT3svdL643n5d8YSlxDPAye0nBHrfv
Nyf9LLc32HRg4mnRmSVLlsTFxU

Elapsed: 0.00 min


### Relación con Yang, Kim y Song (2020)

Yang, S., Kim, J. K., & Song, R. (2020). Doubly Robust Inference when Combining Probability and
Non-Probability Samples with High Dimensional Data. *JRSS B*, **82**(2), 445–465.
[doi:10.1111/rssb.12354](https://doi.org/10.1111/rssb.12354).

El artículo combina una muestra probabilística de referencia con otra no probabilística que observa
el desenlace. Combina un modelo de participación con uno del desenlace; bajo sus supuestos,
la consistencia puede mantenerse si uno de los dos está correctamente especificado. Su procedimiento
incluye selección penalizada de variables y estimación posterior orientada a reducir sesgo.
Requiere participación positiva y selección ignorable condicionada a las covariables; la doble robustez
no sustituye esos supuestos. Su estimador aumentado tiene la forma:

\[
\widehat\mu_{DR}=N^{-1}\left\{\sum_{i\in A}d_i\widehat m(X_i)
 +\sum_{i\in B}\frac{Y_i-\widehat m(X_i)}{\widehat\pi_B(X_i)}\right\}.
\]

**Aplicación a este proyecto.** CASEN puede aportar la distribución de referencia; la ENPG es una encuesta
probabilística con cobertura restringida, por lo que no equivale sin adaptación a la muestra B del artículo.
Incluir ruralidad explicita la falta de soporte ENPG rural; omitirla permite calcular pesos, pero impone
el supuesto no comprobado de transporte dentro de sexo × edad × educación.

| Componente | Situación y consecuencia |
|---|---|
| Covariables compartidas | Sexo, edad y educación armonizados; consumo no medido aquí en CASEN. |
| Positividad nacional | Falta soporte observado rural en ENPG; una regresión no crea observaciones rurales. |
| Años intermedios | Las metas interpoladas no son una muestra probabilística observada de ese año. |
| Modelos de selección y resultado | No se implementa la selección penalizada ni la inferencia de alta dimensión de Yang. |
| Incertidumbre | Pendiente propagación conjunta de diseños, modelos, no respuesta e interpolación. |
| Paso defendible | Evaluar primero transporte dentro del marco cubierto y obtener ruralidad EPS oficial con consumo comparable antes de extrapolar. |

Como comprobación algebraica, la expresión aumentada con pesos que calibran exactamente las mismas celdas
y un predictor constante por celda devuelve la postestratificación. Aquí se usa un predictor constante global
para verificar esa identidad. **La coincidencia no es una prueba de doble robustez ni una implementación del método
de Yang**: ambas expresiones reutilizan la misma información. Se conserva su diferencia numérica en el CSV.


In [9]:
.t0 <- Sys.time()
identity_check <- read_result("rural_augmented_identity")
balance_check <- read_result("rural_balance")
stopifnot(nrow(identity_check) == 112, all(identity_check$abs_difference < 1e-10),
  !any(identity_check$is_yang_high_dimensional_fit), max(abs(balance_check$smd_after)) < 1e-10)
eca_show_table(identity_check |> dplyr::group_by(anio) |>
  dplyr::summarise(max_difference = max(abs_difference), .groups = "drop"),
  "Identidad aumentada: comprobación algebraica, no validación de doble robustez", digits = 12)
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


anio,max_difference
2012,0
2014,0
2016,0
2018,0
2020,0
2022,0
2024,0


Elapsed: 0.00 min


## 7. Uso y límites de `eps_reid.R`

Se preserva el script original. Su búsqueda automática de sexo/edad falla con las etiquetas actuales de EPS VIII.
Se ejecuta `eps_reid_casen_aps.R`, una copia con selección explícita de `a8/a9` del módulo del entrevistado
y lectura portable del CSV INE. Se preservan **todos los candidatos**, asignaciones, alternativas y márgenes;
su etiqueta de confianza se renombra para aclarar que no es una probabilidad validada. La asignación utiliza
tamaño, perfil sexo–edad, región laboral y respuesta «posta rural». Usar luego esa misma respuesta para validar
una tasa de APS generaría circularidad. La asignación ecológica tampoco observa residencia ni ruralidad individual.

Las asociaciones rurales previas se conservan como referencia histórica, con bandera `models_reexecuted=FALSE`.
Un IC que incluye uno no prueba equivalencia urbano–rural ni que la diferencia sea pequeña.
No se aplica un multiplicador rural al modelo base ni se unen personas EPS a comunas como dato confirmado.

La explicación también usa `rg_T2_gaps.csv`: diferencias entre áreas 100% rurales y 100% urbanas,
por sexo y educación, con control regional. Son modelos anteriores que suponen la misma diferencia
para las edades; no demuestran un efecto rural específico de 30–44 años. `rg_T3_enpg_bias_cells.csv`
aporta n descriptivos de 18–65 años; no un tamaño efectivo ni una validación de esos modelos.
Estas tablas no aportan un efecto rural para lifetime abstainer ni las categorías WHO vigentes.
“Consumo actual” EPS requiere armonización antes de equipararlo al último mes ENPG.


In [10]:
.t0 <- Sys.time()
if (rebuild) eca_eps_audit()
eca_show_table(read_result("eps_script_audit"), "Auditoría del script EPS")
eca_show_table(read_result("eps_rural_historical_reference"), "Referencia rural histórica: modelos no reestimados")
eps_gaps <- utils::read.csv(here::here("__andres_control", "rg_T2_gaps.csv"))
eca_show_table(eps_gaps[eps_gaps$exposure == "rural_asg", ],
  "EPS histórica: diferencias entre áreas por sexo y educación, comunes a las edades; no reestimadas", digits = 3)
eps <- read_result("eps_heuristic_assignment_audit")
eca_show_table(as.data.frame(table(eps$heuristic_label_not_probability)), "Asignaciones EPS por etiqueta heurística", digits = 0)
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


source,md5,adaptation,candidates,assignments,negative_margins,use_for_base_weights
__andres_control/eps_reid_casen_aps.R,e2198a9d4029e120d5ee166b9c83333b,Original discovery fails on current VIII labels; explicitly select respondent a8/a9,3130,77,26,FALSE


outcome,waves,exposure,region_FE,RR,lo95,hi95,p,models_reexecuted,role
Current drinker,VIII,rural_asg,TRUE,0.668,0.421,1.059,0.086,FALSE,Historical ecological sensitivity only; not a transport factor
Grams/day (population),VIII,rural_asg,TRUE,0.961,0.431,2.144,0.923,FALSE,Historical ecological sensitivity only; not a transport factor
Risky (>=20 g W / >=40 g M),VIII,rural_asg,TRUE,0.277,0.058,1.334,0.109,FALSE,Historical ecological sensitivity only; not a transport factor
HED 5+/4+ (population),VIII,rural_asg,TRUE,1.308,0.572,2.989,0.524,FALSE,Historical ecological sensitivity only; not a transport factor


exposure,outcome,sex,edu,delta,lo95,hi95
rural_asg,Current drinker,Men,Primary completed or less,0.113,-0.173,0.399
rural_asg,Current drinker,Men,Secondary completed or less,-0.081,-0.284,0.122
rural_asg,Current drinker,Men,Tertiary completed or higher,-0.189,-0.464,0.087
rural_asg,Current drinker,Women,Primary completed or less,-0.047,-0.299,0.205
rural_asg,Current drinker,Women,Secondary completed or less,-0.241,-0.419,-0.063
rural_asg,Current drinker,Women,Tertiary completed or higher,-0.349,-0.598,-0.100
rural_asg,Grams/day (population),Men,Primary completed or less,2.872,-3.342,9.087
rural_asg,Grams/day (population),Men,Secondary completed or less,2.793,-3.550,9.136
rural_asg,Grams/day (population),Men,Tertiary completed or higher,-0.874,-6.113,4.364
rural_asg,Grams/day (population),Women,Primary completed or less,-1.015,-6.239,4.209


Var1,Freq
alta,13
baja,57
media,7


Elapsed: 0.00 min


## 8. Contrato de integración y comprobaciones

Las tablas objetivo y factores son agregados en CSV. Los pesos individuales se leen así, sin exportar filas:

```r
weights <- readRDS(acc_data("__andres_control/enpg_casen_aps_outputs/enpg_poststrat_nacional.rds.tar.xz.enc"))
```

Usar `peso_educ` si se desea preservar sexo–edad ENPG; `peso_nacional` si se desea la distribución sexo–edad CASEN.
Elegir uno y reconstruir las metas del motor, sin multiplicar ambos ajustes otra vez. Los factores no corrigen
subdeclaración, falta de respuesta dependiente del consumo o diferencias rurales residuales.

La arquitectura de referencia del proyecto sigue siendo Kilian et al. (2025), DOI
[10.1016/S2468-2667(25)00165-3](https://doi.org/10.1016/S2468-2667(25)00165-3), y SIMAH 0.1.1,
[10.5281/zenodo.15641639](https://doi.org/10.5281/zenodo.15641639). Aquí no se implementa ni valida su ciclo anual.


In [11]:
.t0 <- Sys.time()
eca_tests()
f <- read_result("enpg_factors_educ3")
stopifnot(all(abs(f$p_after[f$p_casen > 0] - f$p_casen[f$p_casen > 0]) < 1e-8),
  !anyDuplicated(contact[c("anio", "comuna", "sexo", "ag")]),
  all(health$p >= 0 & health$p <= 1, na.rm = TRUE))
cat("PASS: target shares, preserved totals, sex-age national shares, commune keys and probability bounds\n")
print(sessionInfo())
message(sprintf("Elapsed: %.2f min", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


PASS: education, missing completion, missing education, totals, duplicate targets, unsupported cells
Elapsed: 0.00 min
PASS: target shares, preserved totals, sex-age national shares, commune keys and probability bounds
R version 4.4.1 (2024-06-14 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 26300)

Matrix products: default


locale:
[1] LC_COLLATE=C                        LC_CTYPE=English_United States.utf8
[3] LC_MONETARY=C                       LC_NUMERIC=C                       
[5] LC_TIME=C                          

time zone: America/Santiago
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices datasets  utils     methods   base     

loaded via a namespace (and not attached):
 [1] digest_0.6.39    IRdisplay_1.1    R6_2.6.1         base64enc_0.1-6 
 [5] fastmap_1.2.0    xfun_0.57        tidyselect_1.2.1 magrittr_2.0.5  
 [9] glue_1.8.1       tibble_3.3.1     knitr_1.51       pkgconfig_2.0.3 
[13] htmltools_0.5.9  gener